* parameters of LLM is porportional to amount of training data that can be used to enhance model
* Training time scaling & inference scaling.
    How can you scale both with smaller models and different reasoning techniques
    

Tokens
* middle ground for learning at the character and word level.
* Able to to be fast without omitting "rare words" and elegantly handles word stems.
* For common words, 1 token maps to one word -> Tokens can also represent the start of a word and the fragement (middle) for more complex words i.e., Hand_crafted = 2 tokens, witch_craft = 2
* 1 token = ~3 numbers


https://platform.openai.com/tokenizer

## Tokens

In [1]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4o-mini")

tokens = encoding.encode("Hello, my name is Alexander")

In [2]:
# looks up the id for each token in its vocab library
tokens

[13225, 11, 922, 1308, 382, 34569]

In [3]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f'{token_id} = {token_text}')

13225 = Hello
11 = ,
922 =  my
1308 =  name
382 =  is
34569 =  Alexander


In [4]:
encoding.decode([34569])

' Alexander'

## Illusion of Memory

In [6]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

api_key=os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No key found")
elif not api_key.startswith("sk-proj"):
    print('Key found but has invlaid format. Should start with "sk-proj"')
else:
    print("Key is valid")






Key is valid


In [7]:
from openai import OpenAI

openai = OpenAI()


In [9]:
messages = [
    {'role': 'system', 'content': 'You are a heldful assistant'},
    {'role':'user','content': 'Hi, my name is Alexander'}
]
response = openai.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    temperature=0
)

response.choices[0].message.content

'Hi Alexander! How can I assist you today?'

In [10]:
messages = [
    {'role': 'system', 'content': 'You are a heldful assistant'},
    {'role':'user','content': 'What is my name?'}
]
response = openai.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    temperature=0
)

response.choices[0].message.content

"I'm sorry, but I don't have access to personal information about you unless you share it with me. How can I assist you today?"

In [ ]:
#demonstrates that these messages are stateless

messages = [
    {'role': 'system', 'content': 'You are a heldful assistant'},
    {'role':'user','content': 'Hi, my name is Alexander'},
    {'role': 'assistant', 'content': 'Hi Alexander, how can I help you today?'},
    {'role':'user','content': 'What is my name?'}
]
response = openai.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    temperature=0
)

response.choices[0].message.content

'Your name is Alexander. How can I assist you further?'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

## Considerations

Context Window - Max tokens a model can consider when generating the next token.

This includes the conversation so far and each word that is generated i.e., Words are genrated iteratively with the previous words being added to the context window.
* Input Prompt
* subsequent conversation
* output prompts and tokens

API cos (charge per API Call) = input tokens + output tokens

Caching allows us to store memory to reduce input costs

Check out Vellum leaderboard!!!